In [0]:
print("Hello Spark")

Hello Spark


In [0]:
spark.version

'4.1.0'

In [0]:
# Distributed processing 
# Python can hadle upto few millions of records
# Spark can handle upto few billions of records so using spark is better and scalable, mostly for distributed processing.

In [0]:
# Reading CSV files

events_df = spark.read.csv(
    "/Volumes/workspace/default/week5data/events.csv",
    header=True,
    inferSchema=True
)

users_df = spark.read.csv(
    "/Volumes/workspace/default/week5data/users.csv",
    header=True,
    inferSchema=True
)

events_df.show()

users_df.show()

+--------+-------+----------+-------+-------+----------+----------------+
|event_id|user_id|event_type| device|country|event_date|session_time_sec|
+--------+-------+----------+-------+-------+----------+----------------+
|    1001|    U01| page_view| mobile|  India|  1/1/2024|              45|
|    1002|    U02|     click|desktop|    USA|  1/1/2024|             120|
|    1003|    U01|  purchase| mobile|  India|  1/2/2024|             300|
|    1004|    U03| page_view| tablet|     UK|  1/2/2024|              60|
|    1005|    U02|     click|desktop|    USA|  1/3/2024|              90|
|    1006|   NULL| page_view| mobile|  India|  1/3/2024|              30|
+--------+-------+----------+-------+-------+----------+----------------+

+-------+---------+--------+------------+
|user_id|user_name| segment|created_date|
+-------+---------+--------+------------+
|    U01|     Ravi| premium|  2023-06-01|
|    U02|     Asha|standard|  2023-08-15|
|    U03|    Imran| premium|  2023-09-20|
+------

In [0]:
#printSchema() shows column names and datatypes
events_df.printSchema()

users_df.printSchema()

root
 |-- event_id: integer (nullable = true)
 |-- user_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- device: string (nullable = true)
 |-- country: string (nullable = true)
 |-- event_date: string (nullable = true)
 |-- session_time_sec: integer (nullable = true)

root
 |-- user_id: string (nullable = true)
 |-- user_name: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- created_date: date (nullable = true)



In [0]:
# FILTER 
# filtewr() keeps only the rows that are matching the condition
india_events = events_df.filter(
    events_df.country == "India"
)

india_events.show()

+--------+-------+----------+------+-------+----------+----------------+
|event_id|user_id|event_type|device|country|event_date|session_time_sec|
+--------+-------+----------+------+-------+----------+----------------+
|    1001|    U01| page_view|mobile|  India|  1/1/2024|              45|
|    1003|    U01|  purchase|mobile|  India|  1/2/2024|             300|
|    1006|   NULL| page_view|mobile|  India|  1/3/2024|              30|
+--------+-------+----------+------+-------+----------+----------------+



In [0]:
#Lazy Evaluation
#Transformations do not execute immediately when they were written.
#Execution starts only when an action is called.


# Reading CSV
df = spark.read.csv(
    "/Volumes/workspace/default/week5data/events.csv",
    header=True,
    inferSchema=True
)

# These lines are transformations - nothing runs yet
filtered = df.filter(df.country == 'India')

selected = filtered.select(
    'user_id',
    'event_type',
    'event_date'
)

grouped = selected.groupBy('event_type').count()

# This is an action - Spark now runs ALL the above
grouped.show()

# Another action - Spark runs the pipeline again
total = grouped.count()

print(total)



+----------+-----+
|event_type|count|
+----------+-----+
| page_view|    2|
|  purchase|    1|
+----------+-----+

2


In [0]:
#withColumn() adds or modifies columns
from pyspark.sql import functions as F

events_df.withColumn(
    "session_minutes",
    F.col("session_time_sec") / 60 # adding the minutes column by converting it to minutes
).show()

+--------+-------+----------+-------+-------+----------+----------------+---------------+
|event_id|user_id|event_type| device|country|event_date|session_time_sec|session_minutes|
+--------+-------+----------+-------+-------+----------+----------------+---------------+
|    1001|    U01| page_view| mobile|  India|  1/1/2024|              45|           0.75|
|    1002|    U02|     click|desktop|    USA|  1/1/2024|             120|            2.0|
|    1003|    U01|  purchase| mobile|  India|  1/2/2024|             300|            5.0|
|    1004|    U03| page_view| tablet|     UK|  1/2/2024|              60|            1.0|
|    1005|    U02|     click|desktop|    USA|  1/3/2024|              90|            1.5|
|    1006|   NULL| page_view| mobile|  India|  1/3/2024|              30|            0.5|
+--------+-------+----------+-------+-------+----------+----------------+---------------+



In [0]:
# JOIN
# Join combines the data from multiple dataframeds 
joined_df = events_df.join(
    users_df,
    on="user_id",
    how="left"
)

joined_df.show()

+-------+--------+----------+-------+-------+----------+----------------+---------+--------+------------+
|user_id|event_id|event_type| device|country|event_date|session_time_sec|user_name| segment|created_date|
+-------+--------+----------+-------+-------+----------+----------------+---------+--------+------------+
|    U01|    1001| page_view| mobile|  India|  1/1/2024|              45|     Ravi| premium|  2023-06-01|
|    U02|    1002|     click|desktop|    USA|  1/1/2024|             120|     Asha|standard|  2023-08-15|
|    U01|    1003|  purchase| mobile|  India|  1/2/2024|             300|     Ravi| premium|  2023-06-01|
|    U03|    1004| page_view| tablet|     UK|  1/2/2024|              60|    Imran| premium|  2023-09-20|
|    U02|    1005|     click|desktop|    USA|  1/3/2024|              90|     Asha|standard|  2023-08-15|
|   NULL|    1006| page_view| mobile|  India|  1/3/2024|              30|     NULL|    NULL|        NULL|
+-------+--------+----------+-------+-------+-

In [0]:
# groupBy and aggregation
# aggregation are used to make the calculations on the grouped data
# groupBy() groups the data based on the column name

daily_summary = df.groupBy('country', 'event_date').agg(
    F.count('event_id').alias('event_count'),
    F.avg('session_time_sec').alias('avg_session_sec')
).show()

+-------+----------+-----------+---------------+
|country|event_date|event_count|avg_session_sec|
+-------+----------+-----------+---------------+
|     UK|  1/2/2024|          1|           60.0|
|  India|  1/3/2024|          1|           30.0|
|  India|  1/1/2024|          1|           45.0|
|  India|  1/2/2024|          1|          300.0|
|    USA|  1/3/2024|          1|           90.0|
|    USA|  1/1/2024|          1|          120.0|
+-------+----------+-----------+---------------+



In [0]:

#NULL handiling
# nulls are the missing values in the data
# dropna() drops the rows with null values
# fillna() fills the null values with the given value
events_df.fillna({
    "user_id": "anonymous"
}).show()

+--------+---------+----------+-------+-------+----------+----------------+
|event_id|  user_id|event_type| device|country|event_date|session_time_sec|
+--------+---------+----------+-------+-------+----------+----------------+
|    1001|      U01| page_view| mobile|  India|  1/1/2024|              45|
|    1002|      U02|     click|desktop|    USA|  1/1/2024|             120|
|    1003|      U01|  purchase| mobile|  India|  1/2/2024|             300|
|    1004|      U03| page_view| tablet|     UK|  1/2/2024|              60|
|    1005|      U02|     click|desktop|    USA|  1/3/2024|              90|
|    1006|anonymous| page_view| mobile|  India|  1/3/2024|              30|
+--------+---------+----------+-------+-------+----------+----------------+



In [0]:
#dropna()
events_df.dropna(
    subset=["event_id", "event_type"]
).show()

+--------+-------+----------+-------+-------+----------+----------------+
|event_id|user_id|event_type| device|country|event_date|session_time_sec|
+--------+-------+----------+-------+-------+----------+----------------+
|    1001|    U01| page_view| mobile|  India|  1/1/2024|              45|
|    1002|    U02|     click|desktop|    USA|  1/1/2024|             120|
|    1003|    U01|  purchase| mobile|  India|  1/2/2024|             300|
|    1004|    U03| page_view| tablet|     UK|  1/2/2024|              60|
|    1005|    U02|     click|desktop|    USA|  1/3/2024|              90|
|    1006|   NULL| page_view| mobile|  India|  1/3/2024|              30|
+--------+-------+----------+-------+-------+----------+----------------+



In [0]:
# Explicit Schema Definition
#Explicit schema improves consistency and avoids incorrect datatype inference.
from pyspark.sql.types import *

schema = StructType([
    StructField("event_id", IntegerType(), True),
    StructField("user_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("device", StringType(), True),
    StructField("country", StringType(), True),
    StructField("event_date", StringType(), True),
    StructField("session_time_sec", IntegerType(), True)
])

events_schema_df = spark.read.csv(
    "/Volumes/workspace/default/week5data/events.csv",
    header=True,
    schema=schema
)

events_schema_df.printSchema()

root
 |-- event_id: integer (nullable = true)
 |-- user_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- device: string (nullable = true)
 |-- country: string (nullable = true)
 |-- event_date: string (nullable = true)
 |-- session_time_sec: integer (nullable = true)



In [0]:
# Partitions, Repartition, Coalesce, and Cache

# Repartition: increases or redistributes partitions
df_repartitioned = df.repartition(4)

# Coalesce: reduces partitions
df_smaller = df_repartitioned.coalesce(2)

# Reuse DataFrame without cache
india_events = df.filter(df.country == "India")

india_events.show()

+--------+-------+----------+------+-------+----------+----------------+
|event_id|user_id|event_type|device|country|event_date|session_time_sec|
+--------+-------+----------+------+-------+----------+----------------+
|    1001|    U01| page_view|mobile|  India|  1/1/2024|              45|
|    1003|    U01|  purchase|mobile|  India|  1/2/2024|             300|
|    1006|   NULL| page_view|mobile|  India|  1/3/2024|              30|
+--------+-------+----------+------+-------+----------+----------------+



In [0]:
# Mini Project: Streaming Media Clickstream Pipeline
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Step 1: Define Explicit Schemas
events_schema = StructType([
    StructField("event_id", IntegerType(), True),
    StructField("user_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("device", StringType(), True),
    StructField("country", StringType(), True),
    StructField("event_date", StringType(), True),
    StructField("session_time_sec", IntegerType(), True)
])

users_schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("user_name", StringType(), True),
    StructField("segment", StringType(), True),
    StructField("created_date", StringType(), True)
])

# Step 2: Read Raw CSV Files
# The data arrives as raw CSV files.
# We read events and users data using explicit schemas.

events_df = spark.read.csv(
    "/Volumes/workspace/default/week5data/events.csv",
    header=True,
    schema=events_schema
)

users_df = spark.read.csv(
    "/Volumes/workspace/default/week5data/users.csv",
    header=True,
    schema=users_schema
)

events_df.show()
users_df.show()

# Step 3: Validate NULL Values
# We count NULL user_id and NULL event_type rows.
# These counts help us understand data quality issues.

null_user_id_count = events_df.filter(
    F.col("user_id").isNull()
).count()

null_event_type_count = events_df.filter(
    F.col("event_type").isNull()
).count()

print("NULL user_id count:", null_user_id_count)
print("NULL event_type count:", null_event_type_count)

# Step 4: Separate Bad Records
# event_id and event_type are important fields.
# If either is NULL, the record is not useful for reliable reporting.
# We route those rows to a separate bad_records DataFrame.

bad_records_df = events_df.filter(
    F.col("event_id").isNull() | F.col("event_type").isNull()
)

valid_events_df = events_df.filter(
    F.col("event_id").isNotNull() & F.col("event_type").isNotNull()
)

bad_records_df.show()

# Step 5: Fill NULL user_id with anonymous
# In clickstream data, missing user_id may represent anonymous users.
# These records are still useful for traffic analytics.

clean_events_df = valid_events_df.fillna({
    "user_id": "anonymous"
})

clean_events_df.show()

# Step 6: Join Events with Users
# We use a left join so that all event records are kept.
# Even if user details are missing, the event should remain in analysis.

joined_df = clean_events_df.join(
    users_df,
    on="user_id",
    how="left"
)

joined_df.show()


# Step 7: Add session_minutes Column
# session_time_sec is converted into minutes for easier reporting.

enriched_df = joined_df.withColumn(
    "session_minutes",
    F.col("session_time_sec") / 60
)

enriched_df.show()

# Step 8: Aggregate Daily User Activity
# We group by user_id, country, and event_date.
# We calculate:

daily_summary_df = enriched_df.groupBy(
    "user_id",
    "country",
    "event_date"
).agg(
    F.count("event_id").alias("event_count"),
    F.sum("session_minutes").alias("total_session_minutes"),
    F.avg("session_minutes").alias("avg_session_minutes"),
    F.collect_set("device").alias("device_types")
)

daily_summary_df.show(truncate=False)

# Step 9: Write Curated Output as Parquet
# Parquet is used because it is columnar, compressed, and faster for analytics.
# Partitioning by event_date helps when analysts filter data by date.

daily_summary_df.write.mode("overwrite").partitionBy(
    "event_date"
).parquet(
    "/Volumes/workspace/default/week5data/output/curated_daily_user_activity"
)

# Step 10: Write Bad Records Separately
bad_records_df.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/week5data/output/bad_records"
)

# If clean_events_df or enriched_df is reused multiple times,
# cache() can improve performance by storing it in memory, But in Databricks Serverless, cache may not be supported.
# Overusing cache can also create memory pressure.

+--------+-------+----------+-------+-------+----------+----------------+
|event_id|user_id|event_type| device|country|event_date|session_time_sec|
+--------+-------+----------+-------+-------+----------+----------------+
|    1001|    U01| page_view| mobile|  India|  1/1/2024|              45|
|    1002|    U02|     click|desktop|    USA|  1/1/2024|             120|
|    1003|    U01|  purchase| mobile|  India|  1/2/2024|             300|
|    1004|    U03| page_view| tablet|     UK|  1/2/2024|              60|
|    1005|    U02|     click|desktop|    USA|  1/3/2024|              90|
|    1006|   NULL| page_view| mobile|  India|  1/3/2024|              30|
+--------+-------+----------+-------+-------+----------+----------------+

+-------+---------+--------+------------+
|user_id|user_name| segment|created_date|
+-------+---------+--------+------------+
|    U01|     Ravi| premium|    6/1/2023|
|    U02|     Asha|standard|   8/15/2023|
|    U03|    Imran| premium|   9/20/2023|
+------

In [0]:
events (clickstream data)
event_id | user_id | event_type  | device   | country | event_date  | session_time_sec
1001     | U01     | page_view   | mobile   | India   | 2024-01-01  | 45
1002     | U02     | click       | desktop  | USA     | 2024-01-01  | 120
1003     | U01     | purchase    | mobile   | India   | 2024-01-02  | 300
1004     | U03     | page_view   | tablet   | UK      | 2024-01-02  | 60
1005     | U02     | click       | desktop  | USA     | 2024-01-03  | 90
1006     | NULL    | page_view   | mobile   | India   | 2024-01-03  | 30

users
user_id | user_name | segment   | created_date
U01     | Ravi      | premium   | 2023-06-01
U02     | Asha      | standard  | 2023-08-15
U03     | Imran     | premium   | 2023-09-20
